In [335]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
# Linear models
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Support Vector Regression
from sklearn.svm import SVR

# Tree-based models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

# Neural Network
from sklearn.neural_network import MLPRegressor

# XGBoost
from xgboost import XGBRegressor

In [336]:
df = pd.read_csv('../data/processed/final_data_set.csv')

In [337]:
x = df.drop(columns=['price'])
y = df['price']

In [338]:
# Applying the log1p transformation to the target variable
y_log_transformed = np.log1p(y)

In [132]:
columns_to_encode = ['property_type', 'sector', 'balcony', 'additionalRoom', 'agePossession', 'luxury_category', 'floor_cat']

In [133]:
columns_to_encode

['property_type',
 'sector',
 'balcony',
 'additionalRoom',
 'agePossession',
 'luxury_category',
 'floor_cat']

In [134]:
df.head()

,Unnamed: 0,property_type,sector,price,bedRoom,bathroom,balcony,additionalRoom,agePossession,built_up_area,luxury_category,bhk,floor_cat
0,0,flat,sector 49,0.35,2,2,2,not_available,0-1 Years Old,616.0,Unfurnished,2,higher
1,1,flat,sector 49,0.38,2,2,2,not_available,1-5 Years Old,750.0,Basic,2,higher
2,2,flat,sector 92,0.49,2,2,2,not_available,1-5 Years Old,1109.0,Unfurnished,2,lower
3,3,flat,sector 89,0.80,2,3,3,"study room,others",Under Construction,1660.0,Unfurnished,2,middle
4,4,flat,sector 49,1.55,3,2,3+,not_available,Upcoming,1730.0,Unfurnished,3,middle


In [135]:
df.drop(columns=['Unnamed: 0'])

,property_type,sector,price,bedRoom,bathroom,balcony,additionalRoom,agePossession,built_up_area,luxury_category,bhk,floor_cat
0,flat,sector 49,0.35,2,2,2,not_available,0-1 Years Old,616.0,Unfurnished,2,higher
1,flat,sector 49,0.38,2,2,2,not_available,1-5 Years Old,750.0,Basic,2,higher
2,flat,sector 92,0.49,2,2,2,not_available,1-5 Years Old,1109.0,Unfurnished,2,lower
3,flat,sector 89,0.80,2,3,3,"study room,others",Under Construction,1660.0,Unfurnished,2,middle
4,flat,sector 49,1.55,3,2,3+,not_available,Upcoming,1730.0,Unfurnished,3,middle
...,...,...,...,...,...,...,...,...,...,...,...,...
3210,flat,sector 109,2.20,3,4,3,servant room,1-5 Years Old,1860.0,Premium,3,middle
3211,flat,sector 62,7.56,4,4,3+,"pooja room,servant room",1-5 Years Old,4200.0,Premium,4,higher
3212,flat,sector 33,0.72,3,2,3,not_available,Under Construction,1081.0,Unfurnished,3,lower
3213,flat,sector 79,1.40,2,2,2,"study room,servant room,store room",1-5 Years Old,1556.0,Premium,2,lower


In [136]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

numerical_cols = [
    'bedRoom',
    'bathroom',
    'built_up_area',
    'bhk'
]

columns_to_encode = [
    'property_type',
    'sector',
    'balcony',
    'additionalRoom',
    'agePossession',
    'luxury_category',
    'floor_cat'
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numerical_cols
        ),

        (
            'cat',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            columns_to_encode
        )
    ],
    remainder='drop'
)

In [137]:
# made pipeline 
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [138]:
# K-fold cross-validation
kfold = KFold(n_splits=40, shuffle=True, random_state=64)
scores = cross_val_score(pipeline, x, y_log_transformed, cv=kfold, scoring='r2')

scores.mean()

np.float64(0.6491629587226143)

In [139]:
scores.mean(),scores.std()

(np.float64(0.6491629587226143), np.float64(0.0666452431106777))

In [140]:
X_train, X_test, y_train, y_test = train_test_split(x,y_log_transformed,test_size=0.2,random_state=42)

In [141]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [142]:
y_pred = pipeline.predict(X_test) 

In [143]:
y_pred = np.expm1(y_pred) # while traing the dataset using log transformation then change the  rela price

In [144]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.691539860124678

In [145]:
def scorer(model_name , model):
    output = []
    output.append(model_name)
    pipeline = Pipeline([
        ('preprocessor' , preprocessor),
        ('regressor' , model)
    ])
    
    # k-fold cross validation
    kfold = KFold(n_splits=10,shuffle=True , random_state=42)
    scores = cross_val_score(pipeline , x,y_log_transformed,cv=kfold , scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(x,y_log_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    output.append(mean_absolute_error(np.expm1(y_test) , y_pred))
    
    return output
                                                         

In [146]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [147]:
model_output = []
for model_name , model in model_dict.items():
    model_output.append(scorer(model_name , model))

In [148]:
model_df = pd.DataFrame(model_output , columns=['name' , 'r2' , 'MAE'])

In [149]:

model_df.sort_values(['MAE'] , inplace=True)
model_df

,name,r2,MAE
10,xgboost,0.857449,0.367397
6,extra trees,0.842038,0.374266
5,random forest,0.846733,0.375259
7,gradient boosting,0.818532,0.455194
4,decision tree,0.700324,0.471091
9,mlp,0.705372,0.588340
1,svr,0.679649,0.624540
2,ridge,0.649859,0.691353
0,linear_reg,0.649850,0.691540
8,adaboost,0.642413,0.735822


# ONE HOT ENCODING

In [150]:
df.head()

,Unnamed: 0,property_type,sector,price,bedRoom,bathroom,balcony,additionalRoom,agePossession,built_up_area,luxury_category,bhk,floor_cat
0,0,flat,sector 49,0.35,2,2,2,not_available,0-1 Years Old,616.0,Unfurnished,2,higher
1,1,flat,sector 49,0.38,2,2,2,not_available,1-5 Years Old,750.0,Basic,2,higher
2,2,flat,sector 92,0.49,2,2,2,not_available,1-5 Years Old,1109.0,Unfurnished,2,lower
3,3,flat,sector 89,0.80,2,3,3,"study room,others",Under Construction,1660.0,Unfurnished,2,middle
4,4,flat,sector 49,1.55,3,2,3+,not_available,Upcoming,1730.0,Unfurnished,3,middle


In [151]:
# Creating a column Transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num' , StandardScaler() , ['bedRoom' , 'bathroom' , 'built_up_area' , 'bhk' ]),
        ('cat' , OrdinalEncoder(handle_unknown='use_encoded_value',
                        unknown_value=-1),[ 'property_type','balcony','additionalRoom','floor_cat']),
        ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['sector','agePossession','luxury_category'])
    ],
    remainder='passthrough'
)

In [152]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor' , preprocessor),
    ('regressor' , LinearRegression())
])

In [153]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, x, y_log_transformed, cv=kfold, scoring='r2')

c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [154]:

scores.mean() ,scores.std()

(np.float64(0.801208541045123), np.float64(0.03928890600292984))

In [155]:
X_train, X_test, y_train, y_test = train_test_split(x,y_log_transformed,test_size=0.2,random_state=42)

In [156]:

pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [157]:
y_pred = pipeline.predict(X_test)

c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [158]:
y_pred = np.expm1(y_pred)

In [159]:

mean_absolute_error(np.expm1(y_test),y_pred)

0.5195523148566419

In [160]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, x, y_log_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(x,y_log_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [161]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [162]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\s

In [163]:
model_df = pd.DataFrame(model_output, columns=['name','r2','MAE'])

In [164]:
model_df

,name,r2,MAE
0,linear_reg,0.801209,0.519552
1,svr,0.012152,1.037301
2,ridge,0.802588,0.515668
3,LASSO,0.033743,1.043045
4,decision tree,0.694427,0.513437
5,random forest,0.823012,0.428821
6,extra trees,0.852155,0.381779
7,gradient boosting,0.805659,0.491209
8,adaboost,0.558101,0.764688
9,mlp,0.673581,0.749017


In [165]:
model_df.sort_values(['MAE'] , inplace=True)
model_df

,name,r2,MAE
6,extra trees,0.852155,0.381779
10,xgboost,0.845551,0.420085
5,random forest,0.823012,0.428821
7,gradient boosting,0.805659,0.491209
4,decision tree,0.694427,0.513437
2,ridge,0.802588,0.515668
0,linear_reg,0.801209,0.519552
9,mlp,0.673581,0.749017
8,adaboost,0.558101,0.764688
1,svr,0.012152,1.037301


### One hotEncoding with PCA


In [229]:

# Creating a column Transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['bedRoom', 'bathroom', 'built_up_area', 'bhk']
        ),

        (
            'cat',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            [
                'property_type',
                'balcony',
                'additionalRoom',
                'floor_cat',
                'luxury_category'
            ]
        ),

        (
            'cat1',
            OneHotEncoder(
                drop='first',
                handle_unknown='ignore'
                ,sparse_output=False
            ),
            ['sector', 'agePossession']
        )
    ],
    remainder='passthrough'
)

In [ ]:

# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95) ),
    ('regressor', LinearRegression())
])

In [231]:

# K-fold cross-validation
kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipeline,
    x,
    y_log_transformed,
    cv=kfold,
    scoring='r2'
)


c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [233]:

scores.mean() , scores.std()

(np.float64(0.7288695671614003), np.float64(0.04814345385560579))

In [234]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=50 ,svd_solver='full')),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
    )

    scores = cross_val_score(
    pipeline,
    x,
    y_log_transformed,
    cv=kfold,
    scoring='r2'
    )
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(x,y_log_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [235]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [236]:

model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\aadarsh kumar verma\anaconda3\Lib\s

In [237]:

model_df = pd.DataFrame(model_output, columns=['name','r2','MAE'])

In [238]:

model_df.sort_values(['MAE'])

,name,r2,MAE
6,extra trees,0.815981,0.430649
5,random forest,0.790641,0.474903
10,xgboost,0.795308,0.489888
7,gradient boosting,0.780076,0.514608
2,ridge,0.728997,0.600869
0,linear_reg,0.728870,0.601460
4,decision tree,0.550107,0.665816
8,adaboost,0.582674,0.688425
9,mlp,0.585474,0.858251
1,svr,0.033254,1.028409


### target Encoder

In [241]:
!pip install category_encoders

In [250]:
df.head()

,Unnamed: 0,property_type,sector,price,bedRoom,bathroom,balcony,additionalRoom,agePossession,built_up_area,luxury_category,bhk,floor_cat
0,0,flat,sector 49,0.35,2,2,2,not_available,0-1 Years Old,616.0,Unfurnished,2,higher
1,1,flat,sector 49,0.38,2,2,2,not_available,1-5 Years Old,750.0,Basic,2,higher
2,2,flat,sector 92,0.49,2,2,2,not_available,1-5 Years Old,1109.0,Unfurnished,2,lower
3,3,flat,sector 89,0.80,2,3,3,"study room,others",Under Construction,1660.0,Unfurnished,2,middle
4,4,flat,sector 49,1.55,3,2,3+,not_available,Upcoming,1730.0,Unfurnished,3,middle


In [270]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'luxury_category', 'floor_cat']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
                    [
                        'property_type',
                        'balcony',
                        'additionalRoom',
                        'floor_cat',
                        'luxury_category'
                    ]
        ),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [271]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [272]:

# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, x, y_log_transformed, cv=kfold, scoring='r2')

In [273]:

scores.mean(),scores.std()

(np.float64(0.7741547632535045), np.float64(0.035364977444618986))

In [277]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, x, y_log_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(x,y_log_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [278]:

model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [279]:

model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [280]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [281]:

model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.856846,0.381033
5,random forest,0.845313,0.410513
10,xgboost,0.835250,0.427222
7,gradient boosting,0.832428,0.455854
4,decision tree,0.720547,0.491058
2,ridge,0.774219,0.544810
0,linear_reg,0.774155,0.544814
8,adaboost,0.711271,0.612840
9,mlp,0.673219,0.728467
1,svr,0.013087,1.036801


## Hyperparameter Tuning

In [285]:
rf_params = {
    'regressor__n_estimators': [200, 400, 600],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['sqrt', 0.7, 1.0]
}

In [298]:
columns_ordinal = [
    'property_type',
    'balcony',
    'luxury_category',
    'floor_cat'
]

columns_onehot = [
    'agePossession'
]

columns_target = [
    'sector'
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['bedRoom', 'bathroom', 'built_up_area']
        ),

        (
            'ordinal',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            columns_ordinal
        ),

        (
            'onehot',
            OneHotEncoder(
                drop='first',
                handle_unknown='ignore',
                sparse_output=False
            ),
            columns_onehot
        ),

        (
            'target',
            ce.TargetEncoder(
                handle_unknown='value',
                handle_missing='value'
            ),
            columns_target
        )
    ],
    remainder='drop'
)

In [299]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

In [300]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [301]:
from sklearn.model_selection import RandomizedSearchCV

rf_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=rf_params,
    n_iter=30,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=4
)

In [302]:
rf_search.fit(x, y_log_transformed)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,Pipeline(step...Regressor())])
,param_distributions,"{'regressor__max_depth': [None, 10, ...], 'regressor__max_features': ['sqrt', 0.7, ...], 'regressor__min_samples_leaf': [1, 2, ...], 'regressor__min_samples_split': [2, 5, ...], ...}"
,n_iter,30
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,4
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [303]:
final_pipe = rf_search.best_estimator_

In [304]:
rf_search.best_params_

{'regressor__n_estimators': 600,
 'regressor__min_samples_split': 2,
 'regressor__min_samples_leaf': 1,
 'regressor__max_features': 0.7,
 'regressor__max_depth': 30}

In [305]:

rf_search.best_score_

np.float64(0.8550418243136395)

In [306]:

final_pipe.fit(x,y_log_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('ordinal', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Exporting the model

In [339]:
x.drop(columns=['Unnamed: 0'] ,inplace=True)

In [340]:

# Creating a column Transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['bedRoom', 'bathroom', 'built_up_area', 'bhk']
        ),

        (
            'cat',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            [
                'property_type',
                'balcony',
                'additionalRoom',
                'floor_cat',
                'luxury_category'
            ]
        ),

        (
            'cat1',
            OneHotEncoder(
                drop='first',
                handle_unknown='ignore'
                ,sparse_output=False
            ),
            ['sector', 'agePossession']
        )
    ],
    remainder='passthrough'
)

In [341]:

best_rf = RandomForestRegressor(
    n_estimators=600,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=0.7,
    max_depth=30,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', best_rf)
])

In [342]:

pipeline.fit(x,y_log_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [343]:
import pickle

with open('../models/model.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

In [344]:
with open('../models/df.pkl', 'wb') as file:
    pickle.dump(x, file)

In [345]:
x

,property_type,sector,bedRoom,bathroom,balcony,additionalRoom,agePossession,built_up_area,luxury_category,bhk,floor_cat
0,flat,sector 49,2,2,2,not_available,0-1 Years Old,616.0,Unfurnished,2,higher
1,flat,sector 49,2,2,2,not_available,1-5 Years Old,750.0,Basic,2,higher
2,flat,sector 92,2,2,2,not_available,1-5 Years Old,1109.0,Unfurnished,2,lower
3,flat,sector 89,2,3,3,"study room,others",Under Construction,1660.0,Unfurnished,2,middle
4,flat,sector 49,3,2,3+,not_available,Upcoming,1730.0,Unfurnished,3,middle
...,...,...,...,...,...,...,...,...,...,...,...
3210,flat,sector 109,3,4,3,servant room,1-5 Years Old,1860.0,Premium,3,middle
3211,flat,sector 62,4,4,3+,"pooja room,servant room",1-5 Years Old,4200.0,Premium,4,higher
3212,flat,sector 33,3,2,3,not_available,Under Construction,1081.0,Unfurnished,3,lower
3213,flat,sector 79,2,2,2,"study room,servant room,store room",1-5 Years Old,1556.0,Premium,2,lower


### try out predication

In [346]:

x.columns

Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'additionalRoom', 'agePossession', 'built_up_area', 'luxury_category',
       'bhk', 'floor_cat'],
      dtype='object')

In [347]:

x.iloc[0].values

array(['flat', 'sector 49', np.int64(2), np.int64(2), '2',
       'not_available', '0-1 Years Old', np.float64(616.0), 'Unfurnished',
       np.int64(2), 'higher'], dtype=object)

In [387]:
data = [['flat', 'sector 44', np.int64(2), np.int64(4), '4',
       'not_available', '0-1 Years Old', np.float64(2000.0), 'Luxury',
       np.int64(4), 'higher']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'additionalRoom', 'agePossession', 'built_up_area', 'luxury_category',
       'bhk', 'floor_cat'] 

In [383]:
x['luxury_category'].value_counts()

luxury_category
Premium        1313
Unfurnished    1206
Basic           309
Semi-Luxury     197
Luxury          190
Name: count, dtype: int64

In [388]:
one_df = pd.DataFrame(data, columns=columns)
one_df

,property_type,sector,bedRoom,bathroom,balcony,additionalRoom,agePossession,built_up_area,luxury_category,bhk,floor_cat
0,flat,sector 44,2,4,4,not_available,0-1 Years Old,2000.0,Luxury,4,higher


In [389]:
np.expm1(pipeline.predict(one_df))

c:\Users\aadarsh kumar verma\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


array([1.66242197])